# 트랜스포머 파라미터 계산과 메모리 예산 - 파라미터 수 정밀 계산

- Tutorial ID: `ull-5`
- Tutorial: 트랜스포머 파라미터 계산과 메모리 예산
- Section ID: `ull-5-1`
- Section: 파라미터 수 정밀 계산


In [ ]:
# ============================================================
# 코드 읽는 법 — 파라미터 수 정밀 계산
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 62)
print("트랜스포머 파라미터 계산과 메모리 예산")
print("=" * 62)

In [ ]:
# 1. 정밀한 파라미터 수 계산

In [ ]:
print("\n1. 파라미터 수 정밀 계산")
print("-" * 50)

def count_parameters(vocab_size, d_model, n_layers, n_heads, d_ff,
                     max_seq=2048, tie_embeddings=True, use_swiglu=False,
                     use_bias=True):
    """트랜스포머 파라미터 수 정밀 계산"""
    params = {}
    
    # 토큰 임베딩
    params['token_embed'] = vocab_size * d_model
    
    # 위치 임베딩 (학습 가능한 경우)
    params['pos_embed'] = max_seq * d_model
    
    # 어텐션 레이어 (per layer)
        d_head = d_model // n_heads
    attn_per_layer = 4 * d_model * d_model  # W_Q, W_K, W_V, W_O
    if use_bias:
        attn_per_layer += 4 * d_model  # 바이어스
    params['attention'] = attn_per_layer * n_layers
    
    # MLP 레이어 (per layer)
    if use_swiglu:
        # SwiGLU: 3 행렬 (gate, up, down), d_ff = 2/3 * 4 * d_model
        mlp_per_layer = 3 * d_model * d_ff
    else:
        mlp_per_layer = 2 * d_model * d_ff  # up + down
    if use_bias:
        mlp_per_layer += d_ff + d_model
    params['mlp'] = mlp_per_layer * n_layers
    
    # 레이어 노름 (2 per layer + 1 final)
    ln_per_layer = 2 * 2 * d_model  # 2 norms × (gamma + beta)
    params['layer_norm'] = ln_per_layer * n_layers + 2 * d_model  # + final
    
    # 언임베딩
    if tie_embeddings:
        params['unembed'] = 0  # 임베딩과 공유
    else:
        params['unembed'] = d_model * vocab_size
    
    return params

# 실제 모델들
models = [
    ("GPT-2 Small",   50257, 768,  12, 12, 3072,  1024, True,  False, True),
    ("GPT-2 Medium",  50257, 1024, 24, 16, 4096,  1024, True,  False, True),
    ("GPT-2 Large",   50257, 1280, 36, 20, 5120,  1024, True,  False, True),
    ("GPT-2 XL",      50257, 1600, 48, 25, 6400,  1024, True,  False, True),
    ("LLaMA-7B",      32000, 4096, 32, 32, 11008, 2048, False, True,  False),
    ("LLaMA-13B",     32000, 5120, 40, 40, 13824, 2048, False, True,  False),
    ("LLaMA-70B",     32000, 8192, 80, 64, 28672, 4096, False, True,  False),
]

print(f"{'Model':>14}  {'Total':>12}  {'Embed':>10}  {'Attn':>10}  {'MLP':>10}  {'LN':>8}")
print("  " + "-" * 72)

for name, V, d, L, H, ff, seq, tie, swi, bias in models:
    p = count_parameters(V, d, L, H, ff, seq, tie, swi, bias)
    total = sum(p.values())
    
    # 공식 파라미터 수와 대략적으로 일치 확인
    print(f"  {name:>12}  {total/1e9:>9.2f}B  {p['token_embed']/1e6:>7.1f}M  "
          f"{p['attention']/1e6:>7.1f}M  {p['mlp']/1e6:>7.1f}M  {p['layer_norm']/1e3:>5.0f}K")

In [ ]:
# 2. 메모리 예산 계산

In [ ]:
print("\n\n2. 훈련 시 메모리 예산")
print("-" * 50)

def training_memory(n_params, batch_size, seq_len, d_model, n_layers,
                    precision='fp32'):
    """훈련 시 총 메모리 사용량 추정"""
    if precision == 'fp32':
        bytes_per_param = 4
    elif precision == 'fp16':
        bytes_per_param = 2
    elif precision == 'bf16':
        bytes_per_param = 2
    else:
        bytes_per_param = 4
    
    mem = {}
    
    # 모델 가중치
    mem['weights'] = n_params * bytes_per_param
    
    # 그래디언트
    mem['gradients'] = n_params * bytes_per_param
    
    # Adam 옵티마이저 상태 (m, v) — 항상 FP32
    mem['optimizer'] = n_params * 4 * 2  # m and v in FP32
    
    # FP32 마스터 가중치 (혼합 정밀도 시)
    if precision in ('fp16', 'bf16'):
        mem['master_weights'] = n_params * 4
    else:
        mem['master_weights'] = 0
    
    # 활성화값 (대략적 추정)
    # 각 레이어: ~10 * batch * seq * d_model * bytes
    mem['activations'] = 10 * batch_size * seq_len * d_model * bytes_per_param * n_layers
    
    return mem

for name, V, d, L, H, ff, seq, tie, swi, bias in models[:4]:  # GPT-2만
    p = count_parameters(V, d, L, H, ff, seq, tie, swi, bias)
    n_params = sum(p.values())
    
    mem = training_memory(n_params, batch_size=4, seq_len=1024, d_model=d, n_layers=L)
    total_gb = sum(mem.values()) / 1e9
    
    print(f"\n  {name} (batch=4, seq=1024, FP32):")
    print(f"    가중치:      {mem['weights']/1e9:.2f} GB")
    print(f"    그래디언트:  {mem['gradients']/1e9:.2f} GB")
    print(f"    옵티마이저:  {mem['optimizer']/1e9:.2f} GB")
    print(f"    활성화값:    {mem['activations']/1e9:.2f} GB")
    print(f"    총 예상:     {total_gb:.2f} GB")

In [ ]:
# 3. 추론 시 메모리

In [ ]:
print("\n\n3. 추론 시 메모리 (KV 캐시 포함)")
print("-" * 50)

for name, V, d, L, H, ff, seq, tie, swi, bias in models:
    n_params = sum(count_parameters(V, d, L, H, ff, seq, tie, swi, bias).values())
        d_head = d // H
    
    # 가중치 (FP16)
    weight_mem = n_params * 2
    
    # KV 캐시 (FP16)
    kv_cache = 2 * L * seq * H * d_head * 2  # 2 for K,V
    
    total = (weight_mem + kv_cache) / 1e9
    print(f"  {name:>12}: 가중치 {weight_mem/1e9:.1f}GB + KV캐시 {kv_cache/1e9:.1f}GB = {total:.1f}GB")